In [3]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
import time
import joblib
from datetime import datetime, timedelta
import csv
import os

import oandapyV20
from oandapyV20.endpoints.accounts import AccountDetails
from oandapyV20.endpoints.pricing import PricingInfo
from bot_functions import *  # Keep if you're using helper functions
from oanda_config import client, OANDA_ACCOUNT_ID  # Handles credentials

# === ✅ CONNECT TO OANDA ===

# Print basic account info
account_request = AccountDetails(accountID=62165237)
account_info = client.request(account_request)

print("\nOANDA Account Info:")
print(f"  Account ID: {account_info['account']['id']}")
print(f"  Balance: {account_info['account']['balance']}")
print(f"  Open Trades: {account_info['account']['openTradeCount']}")
print(f"  Margin Available: {account_info['account']['marginAvailable']}")
print(f"  Currency: {account_info['account']['currency']}")

ModuleNotFoundError: No module named 'oanda_config'

In [ ]:
# ir buscar o modelo
loaded_bundle = joblib.load('eurusd_model.joblib')
print("Bundle loaded.")
# Access individual components from the loaded bundle
model = loaded_bundle['model']
sl_tp_map = loaded_bundle['sl_tp_map']
avg_duration_by_class = loaded_bundle['avg_duration_by_class']
scaler = loaded_bundle['scaler']

#put parameters
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5
mt5.symbol_select(SYMBOL, True)
class_to_direction = {0: -1, 1: -1, 2: 0, 3: 1, 4: 1}
# ver isto...................
DEVIATION = 10
risk_per_trade_percentage = 0.01
threshold = 0.7

# to log the trades
log_file = "trade_log.csv"

# Initialize log file with headers if not exists
if not os.path.exists(log_file):
    with open(log_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "symbol", "direction", "signal", "confidence", "lot_size", "sl", "tp", "profit", "duration_sec", "status"])

# run the model
while True:
    open_positions = mt5.positions_get(symbol=SYMBOL)
    
    if open_positions is None or len(open_positions) == 0:
        account_info = mt5.account_info()
        df = get_latest_data(SYMBOL, TIMEFRAME, 50)
        X = build_dataset(df)
        X_scaled = scale(X, scaler)
        X_treat = create_lstm_sequences(X_scaled, 12)
        last_candle = X_treat[-1:].copy()
        # last_candle = np.reshape(last_candle, (1, last_candle.shape[1], last_candle.shape[2])).astype(np.float32)

        prediction = model.predict(last_candle)
        signal = np.argmax(prediction, axis=1).item()
        confidence = np.max(prediction, axis=1).item()
        sltp = sl_tp_map.get(signal, {'sl': None, 'tp': None})
        sl = sltp['sl']
        tp = sltp['tp']
        avg_duration = avg_duration_by_class.get(signal, 0)
        balance = account_info.equity
        direction = class_to_direction.get(signal, 0)
        print(f"Signal: {signal}, Confidence: {confidence:.2f}")

        if direction == 1 and confidence >= threshold:
            lot_size_multiplier = calculate_lot_size_multiplier(sl, balance, risk_per_trade_percentage)
            if lot_size_multiplier:
                execute_trade(sl, tp, direction, lot_size_multiplier, SYMBOL, DEVIATION)
                entry_time = datetime.now()

                # Log trade entry
                with open(log_file, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        entry_time.isoformat(), SYMBOL, direction, signal, confidence,
                        lot_size_multiplier, sl, tp, "", "", "opened"
                    ])
                print(f"Trade executed at {entry_time}, SL: {sl}, TP: {tp}, Lots: {lot_size_multiplier}")
            else:
                print("Lot size too small, trade skipped.")
                
        else:
            close_trade(avg_duration, open_positions, SYMBOL, DEVIATION)

        time.sleep(300)  # espera 5 minutos

Bundle loaded.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
Signal: 2, Confidence: 0.82
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Signal: 2, Confidence: 0.84
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
Signal: 2, Confidence: 0.85
